In [9]:
import os
import jieba
import numpy as np
import re
from collections import defaultdict

# 定义文件路径
data_path = r"D:\code\natural_language_processing\lab4\lab4\article\article"

# 定义正则表达式去除标点、数字和单字
pattern = re.compile(r'[^\u4e00-\u9fa5]')

# 初始化词频向量字典和词文档集合字典
word_vectors = defaultdict(lambda: np.zeros(20738, dtype=int))
word_docs = defaultdict(set)

# 读取所有文件并进行分词
for i in range(1, 20739):
    file_name = f"{i}.txt"
    file_path = os.path.join(data_path, file_name)
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            # 去除标点、数字和单字
            content = pattern.sub('', content)
            words = jieba.lcut(content)
            words = [word for word in words if len(word) > 1]
            for word in words:
                word_vectors[word][i-1] += 1
                word_docs[word].add(i)

def calculate_euclidean_distance(vec1, vec2):
    return np.linalg.norm(vec1 - vec2)

def find_closest_words(target_word, k):
    if target_word not in word_vectors:
        raise ValueError(f"单词'{target_word}' 没有出现在任何文档中")
    
    target_vector = word_vectors[target_word]
    distances = []
    
    for word, vector in word_vectors.items():
        if word != target_word:
            distance = calculate_euclidean_distance(target_vector, vector)
            distances.append((word, distance))
    
    distances.sort(key=lambda x: x[1])
    closest_words = distances[:k]
    
    # 找出最相似的词及其共同出现的文档
    common_docs = word_docs[target_word]
    for word, _ in closest_words:
        common_docs = common_docs.intersection(word_docs[word])
    
    return closest_words, common_docs

# 接受键盘输入
target_word = input("请输入目标词：")
try:
    k = int(input("请输入要查找的相似词数量："))
    closest_words, common_docs = find_closest_words(target_word, k)
    print(f"最相似的{k}个词：")
    for word, distance in closest_words:
        print(f"{word}")
    print("\n这些词与输入词共同出现的文档:(如果输出 set() 则代表没有共同出现的文档)")
    print(common_docs)
except ValueError as e:
    print(e)

最相似的2个词：
之师
老战士

这些词与输入词共同出现的文档:(如果输出 set() 则代表没有共同出现的文档)
{11399}


In [ ]:
import os
import re
import jieba

# 定义文件路径
input_dir = r'D:\code\natural_language_processing\lab4\lab4\article\article'
output_file = r'D:\code\natural_language_processing\lab4\lab4\dic.txt'
output_result_file = r'D:\code\natural_language_processing\lab4\lab4\output.txt'

# 初始化一个字典来存储词语及其出现的文件
word_dict = {}

# 遍历目录中的所有文件
for filename in sorted(os.listdir(input_dir)):
    if filename.endswith('.txt'):
        file_path = os.path.join(input_dir, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            
            # 去除单字、标点、数字和英语
            content = re.sub(r'[^\u4e00-\u9fa5]', '', content)  # 仅保留中文字符
            content = re.sub(r'\b\w\b', '', content)  # 去除单字
            
            # 使用jieba进行分词
            words = jieba.lcut(content)
            
            # 更新字典
            for word in words:
                if len(word) > 1:  # 确保不是单字
                    if word in word_dict:
                        if filename not in word_dict[word]:
                            word_dict[word].append(filename)
                    else:
                        word_dict[word] = [filename]

# 将结果写入输出文件
with open(output_file, 'w', encoding='utf-8') as out_file:
    for word, files in sorted(word_dict.items()):
        file_str = '->'.join([f.split('.')[0] for f in files])
        out_file.write(f'{word}:{file_str}\n')

print("处理完成，结果已保存到", output_file)

# 定义链表节点
class JumpNode:
    def __init__(self, num):
        self.num = num
        self.next = None
        self.link = None

# 定义跳表节点
class JumpCode:
    def __init__(self, num, link):
        self.num = num
        self.next = None
        self.link = link

# 接受键盘输入的单词
search_word = input("请输入要查询的单词：")
if search_word not in word_dict:
    print(f"错误：单词 '{search_word}' 不在分词结果中。")
else:
    # 接受一个正整数
    try:
        k = int(input("请输入一个范围在1-20738之内的正整数："))
        if k < 1 or k > 20738:
            raise ValueError
    except ValueError:
        print("错误：请输入一个范围在1-20738之内的正整数。")
    else:
        # 构建链表
        file_numbers = [int(f.split('.')[0]) for f in word_dict[search_word]]
        file_numbers.sort()
        
        # 创建链表
        head = None
        current = None
        for num in file_numbers:
            new_node = JumpNode(num)
            if not head:
                head = new_node
                current = new_node
            else:
                current.next = new_node
                current = new_node
        
        # 构建跳表
        jump_head = None
        jump_current = None
        step = int(len(file_numbers) ** 0.5)
        i = 0
        j = 0
        current_node = head
        
        while current_node:
            if not jump_head:
                jump_head = JumpCode(current_node.num, current_node)
                jump_current = jump_head
            else:
                jump_current.next = JumpCode(current_node.num, current_node)
                jump_current = jump_current.next
            
            i += step
            j += 1
            for _ in range(step):
                if current_node.next:
                    current_node = current_node.next
                else:
                    break
        
        # 检查输入的整数k是否在链表范围内
        if k < file_numbers[0] or k > file_numbers[-1]:
            print(f"错误：输入的整数 {k} 不在文档的最小值和最大值的范围内。")
        else:
            # 使用跳表查找
            jump_current = jump_head
            while jump_current.next and jump_current.next.num <= k:
                jump_current = jump_current.next
            
            current_node = jump_current.link
            while current_node and current_node.num < k:
                current_node = current_node.next
            
            if current_node and current_node.num == k:
                print(f"单词 '{search_word}' 出现在文档 {k} 中。")
            else:
                print(f"单词 '{search_word}' 不在文档 {k} 中。")
            
            # 输出链表和跳表到文件
            with open(output_result_file, 'w', encoding='utf-8') as out_file:
                # 输出链表
                current_node = head
                linked_list_str = f"{search_word}:"
                while current_node:
                    linked_list_str += f"{current_node.num}"
                    if current_node.next:
                        linked_list_str += "->"
                    current_node = current_node.next
                out_file.write(linked_list_str + "\n")
                
                # 输出跳表
                jump_current = jump_head
                skip_list_str = f"{search_word}:"
                while jump_current:
                    skip_list_str += f"{jump_current.num}"
                    if jump_current.next:
                        skip_list_str += "->"
                    jump_current = jump_current.next
                out_file.write(skip_list_str + "\n")
            
            print(f"链表和跳表已保存到 {output_result_file}")

处理完成，结果已保存到 D:\code\natural_language_processing\lab4\lab4\dic.txt


KeyboardInterrupt: 

In [1]:
import os
import re
import jieba

# 定义文件路径
input_dir = r'D:\code\natural_language_processing\lab4\lab4\article\article'
output_file = r'D:\code\natural_language_processing\lab4\lab4\dic.txt'
output_result_file = r'D:\code\natural_language_processing\lab4\lab4\output.txt'

# 初始化一个字典来存储词语及其出现的文件
word_dict = {}

# 遍历目录中的所有文件
for filename in sorted(os.listdir(input_dir)):
    if filename.endswith('.txt'):
        file_path = os.path.join(input_dir, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            
            # 去除单字、标点、数字和英语
            content = re.sub(r'[^\u4e00-\u9fa5]', '', content)  # 仅保留中文字符
            content = re.sub(r'\b\w\b', '', content)  # 去除单字
            
            # 使用jieba进行分词
            words = jieba.lcut(content)
            
            # 更新字典
            for word in words:
                if len(word) > 1:  # 确保不是单字
                    if word in word_dict:
                        if filename not in word_dict[word]:
                            word_dict[word].append(filename)
                    else:
                        word_dict[word] = [filename]

# 将结果写入输出文件
with open(output_file, 'w', encoding='utf-8') as out_file:
    for word, files in sorted(word_dict.items()):
        file_str = '->'.join([f.split('.')[0] for f in files])
        out_file.write(f'{word}:{file_str}\n')

print("处理完成，结果已保存到", output_file)

# 定义链表节点
class LinkNode:
    def __init__(self, docID):
        self.docID = docID
        self.next = None

# 定义跳表节点
class SkipNode:
    def __init__(self, docID, link):
        self.docID = docID
        self.next = None
        self.link = link

# 接受键盘输入的单词
search_word = input("请输入要查询的单词：")
if search_word not in word_dict:
    print(f"错误：单词 '{search_word}' 不在分词结果中。")
else:
    # 接受一个正整数
    try:
        k = int(input("请输入一个范围在1-20837之内的正整数："))
        if k < 1 or k > 20837:
            raise ValueError
    except ValueError:
        print("错误：请输入一个范围在1-20837之内的正整数。")
    else:
        # 构建链表
        file_numbers = [int(f.split('.')[0]) for f in word_dict[search_word]]
        file_numbers.sort()
        
        # 创建链表
        head = None
        current = None
        for num in file_numbers:
            new_node = LinkNode(num)
            if not head:
                head = new_node
                current = new_node
            else:
                current.next = new_node
                current = new_node
        
        # 构建跳表
        jump_head = None
        jump_current = None
        step = int(len(file_numbers) ** 0.5)
        i = 0
        j = 0
        current_node = head
        
        while i < len(file_numbers):
            if not jump_head:
                jump_head = SkipNode(current_node.docID, current_node)
                jump_current = jump_head
            else:
                jump_current.next = SkipNode(current_node.docID, current_node)
                jump_current = jump_current.next
            
            i += step
            j += 1
            for _ in range(step):
                if current_node.next:
                    current_node = current_node.next
                else:
                    break
        
        # 设置跳表最后一个节点的next为None
        if jump_current:
            jump_current.next = None
        
        # 检查输入的整数k是否在链表范围内
        if k < file_numbers[0] or k > file_numbers[-1]:
            print(f"错误：输入的整数 {k} 不在文档的最小值和最大值的范围内。")
        else:
            # 使用跳表查找
            jump_current = jump_head
            while jump_current.next and jump_current.next.docID < k:
                jump_current = jump_current.next
            
            current_node = jump_current.link
            while current_node and current_node.docID < k:
                current_node = current_node.next
            
            if current_node and current_node.docID == k:
                print(f"单词 '{search_word}' 出现在文档 {k} 中。")
            else:
                print(f"单词 '{search_word}' 不在文档 {k} 中。")
            
            # 输出链表和跳表到文件
            with open(output_result_file, 'w', encoding='utf-8') as out_file:
                # 输出链表
                current_node = head
                linked_list_str = f"{search_word}:"
                while current_node:
                    linked_list_str += f"{current_node.docID}"
                    if current_node.next:
                        linked_list_str += "->"
                    current_node = current_node.next
                out_file.write(linked_list_str + "\n")
                
                # 输出跳表
                jump_current = jump_head
                skip_list_str = f"{search_word}:"
                while jump_current:
                    skip_list_str += f"{jump_current.docID}"
                    if jump_current.next:
                        skip_list_str += "->"
                    jump_current = jump_current.next
                out_file.write(skip_list_str + "\n")
            
            print(f"链表和跳表已保存到 {output_result_file}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\TANGYI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.503 seconds.
Prefix dict has been built successfully.


处理完成，结果已保存到 D:\code\natural_language_processing\lab4\lab4\dic.txt
单词 '事迹' 出现在文档 10765 中。
链表和跳表已保存到 D:\code\natural_language_processing\lab4\lab4\output.txt
